In [1]:
import os, sys
from pathlib import Path
ROOT = Path().resolve().parent  # notebooks/ -> project root
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import plotly.express as px
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

ROOT      = Path().resolve()
DATA_PATH = ROOT / "data" / "processed"

In [3]:
rrd = pd.read_parquet(DATA_PATH / "rrd_df_novo.parquet")
rrd["nr_candidato"] = rrd["nr_candidato"].astype(str)

# Aliases de compatibilidade
rrd["dias_primeira_receita"]   = rrd["dias_desde_inicio"]
rrd["receita_total_candidato"] = rrd["vr_receita_recursos_partidos"].fillna(0)
rrd["dt_primeira_receita"]     = rrd["dt_receita"]

for col in ["gap_primeira_maior_receita", "share_primeira_receita",
            "maior_receita", "dias_maior_receita"]:
    rrd[col] = np.nan

rrd_df_merge = rrd.copy()

print("Candidatos por ano:")
print(rrd_df_merge.groupby("ano_eleicao").size().rename("n_candidatos").to_frame())
print(f"\nColunas ({len(rrd_df_merge.columns)}): {list(rrd_df_merge.columns)}")

Candidatos por ano:
             n_candidatos
ano_eleicao              
2014                 6178
2018                 7630
2022                 9675

Colunas (35): ['ano_eleicao', 'sg_uf', 'sg_partido', 'nr_candidato', 'nm_candidato', 'ds_sit_tot_turno', 'qt_votos_nominais', 'nr_cpf_candidato', 'n_eleicoes_prefeito', 'n_eleicoes_vereador', 'n_eleicoes_deputado_estadual', 'n_eleicoes_deputado_federal', 'n_eleicoes_governador', 'n_eleicoes_senador', 'vr_receita_outros', 'vr_receita_recursos_partidos', 'vr_receita_fefc', 'vr_receita_fp', 'qe', 'qt_vaga', 'dt_receita', 'fonte_primeiro_repasse', 'dias_desde_inicio', 'prop_votos_nominais_lag', 'alcancou_10pct_qe_hist', 'alcancou_10pct_qe_hist_nom', '10pct_qe_eleicao_atual', 'eleito', 'dias_primeira_receita', 'receita_total_candidato', 'dt_primeira_receita', 'gap_primeira_maior_receita', 'share_primeira_receita', 'maior_receita', 'dias_maior_receita']


## EDA: Timing dos Repasses

In [4]:
px.histogram(
    rrd_df_merge,
    x="dias_primeira_receita",
    color="ano_eleicao",
    barmode="overlay",
    title="Quantos dias de campanha se passaram até o primeiro repasse?",
)

In [5]:
px.histogram(
    rrd_df_merge,
    x="gap_primeira_maior_receita",
    color="ano_eleicao",
    barmode="overlay",
    title="Quantos dias se passaram entre o primeiro repasse e o maior repasse?",
)

In [6]:
rrd_df_merge['gap_primeira_maior_receita'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: gap_primeira_maior_receita, dtype: float64

In [7]:
px.histogram(
    rrd_df_merge,
    x="share_primeira_receita",
    barmode="overlay",
    title="Qual o share da primeira receita em relação ao total arrecadado?",
    nbins=10,
)

In [8]:
rrd_df_merge['share_primeira_receita'].describe()

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: share_primeira_receita, dtype: float64

In [9]:
# eleito aqui é usado apenas como agrupador descritivo — somente candidatos eleitos por QP/Média
rrd_df_merge['eleito_desc'] = np.where(
    rrd_df_merge['ds_sit_tot_turno'].isin(['ELEITO POR QP', 'ELEITO POR MÉDIA']),
    1, 0,
)

In [10]:
rrd_df_merge.groupby('eleito_desc')['dias_maior_receita'].describe()

,count,mean,std,min,25%,50%,75%,max
eleito_desc,,,,,,,,
0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
rrd_df_merge['n_eleicoes_deputado_federal'] = (
    rrd_df_merge['n_eleicoes_deputado_federal'].fillna(0).astype(int)
)
px.box(
    rrd_df_merge.dropna(subset=['dias_maior_receita']),
    x='n_eleicoes_deputado_federal',
    y='dias_maior_receita',
    height=600,
)

In [12]:
px.box(
    rrd_df_merge.dropna(subset=["dias_primeira_receita"]),
    x="n_eleicoes_deputado_federal",
    y="dias_primeira_receita",
)

## Average Marginal Effects — Modelo Fracionário

In [13]:
df_output_modelo = pd.read_parquet(DATA_PATH / "df_ame_baseline.parquet")

df_output_modelo_print = df_output_modelo.loc[
    ~df_output_modelo["variavel"].str.startswith("C")
].copy()

df_output_modelo_print["variavel"] = (
    df_output_modelo_print["variavel"]
    .str.replace("n_eleicoes_deputado_estadual", "Nº Eleições Deputado Estadual")
    .str.replace("n_eleicoes_deputado_federal",  "Nº Eleições Deputado Federal")
    .str.replace("n_eleicoes_senador",            "Nº Eleições Senador")
    .str.replace("n_eleicoes_governador",         "Nº Eleições Governador")
    .str.replace("n_eleicoes_vereador",           "Nº Eleições Vereador")
    .str.replace("n_eleicoes_prefeito",           "Nº Eleições Prefeito")
    .str.replace("np.log(qt_vaga)",               "Ln Magnitude do distrito")
    .str.replace("prop_votos_nominais_lag",        "Proporção de votos nominais intralista t-1")
)

import plotly.graph_objects as go

fig_modelo = px.scatter(
    df_output_modelo_print.sort_values("variavel", ascending=False),
    y="variavel",
    x="dy/dx",
    error_x="Std. Err.",
    labels={"dy/dx": "AME", "variavel": "Variável"},
    template="plotly_white",
    color="ano",
    width=800,
    symbol="ano",
    color_discrete_sequence=["black", "grey"],
    text="dy/dx",
)
fig_modelo.add_vline(x=0, line_dash="dash", line_color="grey")
fig_modelo.update_layout(
    legend=dict(orientation="h", yanchor="bottom", y=1.02,
                xanchor="center", x=0.5, title_text="Eleições"),
    font=dict(size=14),
    margin=dict(l=140, r=40, t=40, b=40),
    width=900, height=500, autosize=False,
)
fig_modelo.update_yaxes(tickfont=dict(size=12))
fig_modelo.update_xaxes(tickfont=dict(size=12))
fig_modelo.update_traces(
    texttemplate="%{text:.2f}",
    textposition="top center",
    marker=dict(size=13),
)
fig_modelo.show()